In [41]:
import argparse
import os
import random,numpy
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

In [42]:
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.utils as vutils
import torchvision.transforms.functional as RF
from torchvision.models.feature_extraction import create_feature_extractor
from PIL import Image
import numpy as np
import random,cv2,pandas,os,numpy
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from sklearn.preprocessing import OneHotEncoder

In [43]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

nz = 100
beta1 = 0.5
lr = 0.0001
batch_size=10
ngpu=1
ngf,nc = 3,3
ndf = 64

device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

Random Seed:  999


In [44]:
train = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/train.csv').drop(columns=['ID','efs_time'])
test = pandas.read_csv('/kaggle/input/equity-post-HCT-survival-predictions/test.csv').drop(columns=['ID'])

In [45]:
encoder = OneHotEncoder(sparse_output=False)

for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]]=encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1))
    else:
        train[i[0]]=train[i[0]]
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]]=numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1)))
    else:
        test[i[0]]=numpy.array(test[i[0]])

In [46]:
train = train.fillna(0)
test = test.fillna(0)

train_y = train['efs']
train_x = train.drop(columns=['efs'])

for i in test:
    test[i]=test[i].to_numpy()

In [47]:
x_ = torch.from_numpy(train_x.to_numpy()).type('torch.FloatTensor')
y_ = torch.from_numpy(train_y.to_numpy()).type('torch.FloatTensor')
sub = torch.from_numpy(test.to_numpy()).type('torch.FloatTensor')

In [48]:
train_x = torch.utils.data.DataLoader(x_,batch_size=batch_size)
train_y = torch.utils.data.DataLoader(y_,batch_size=batch_size)
test_sub = torch.utils.data.DataLoader(sub,batch_size=batch_size)

In [49]:
class ENCODER(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.rafire = nn.Sequential(       
            nn.Linear(57, 1524),
            nn.BatchNorm1d(1524),
            nn.LeakyReLU(),
            
            nn.Linear(1524, 824),
            nn.BatchNorm1d(824),
            nn.LeakyReLU(),
            
            nn.Linear(824, 424),
            nn.BatchNorm1d(424),
            nn.LeakyReLU(),
            
            nn.Linear(424, 824),
            nn.BatchNorm1d(824),
            nn.LeakyReLU(),
            
            nn.Linear(824, 1524),
            nn.BatchNorm1d(1524),
            nn.LeakyReLU(),

            nn.Linear(1524, 3000)
            
        )

    def forward(self,x):
        
        return torch.reshape(self.rafire(x), (10,3,100,100))

encoder = ENCODER().float()
encoder= nn.DataParallel(encoder).to(device)
torch.save(encoder.state_dict(),f'/kaggle/working/encoder_weight.pth')

In [50]:
class EffnetModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        effnet = torchvision.models.efficientnet_v2_m(weights=torchvision.models.efficientnet.EfficientNet_V2_M_Weights.DEFAULT)
        self.model = create_feature_extractor(effnet, ['flatten'])
        self.nn_fracture = torch.nn.Sequential(
            torch.nn.Linear(1280, 2),
        )
    def forward(self, x):
        x = self.model(x)['flatten']
        x = self.nn_fracture(x)
        return x,torch.argmax(x, dim=1)

In [51]:
EFF_NET = high_regressor().float()
EFF_NET= nn.DataParallel(EFF_NET).to(device)

In [52]:
criterion = nn.L1Loss()

optimizer = optim.AdamW(EFF_NET.parameters(), lr=lr,betas=(beta1, 0.999))
scheduler=torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.86)

In [53]:
high_rf,i,j_r,k,z_w_=100000,0,0,0,[]
    
correct,total=0,0

while(True):
    optimizer.zero_grad()
    for data,label in zip(train_x,train_y):
        #z_ = nn.Embedding(10, 3, dtype=torch.float64)
        #print(z_(torch.FloatTensor(data)))
        output = EFF_NET(data.to(device)).view(-1)
        
        err_real = criterion(output, label.to(device))
        err_real.backward()
        optimizer.step()
        #print(err_real.item())
    
    print(f"EPOCH : {i} LOSS_wl : {err_real.item()}")

    if len(z_w_)>=3:
        if len([True for i in range(1,4) if z_w_[len(z_w_)-i]<=z_w_[len(z_w_)-3] and z_w_[len(z_w_)-i]>=z_w_[len(z_w_)-4]])==3:
            z_w_,j_r=[],0
            print(f"lr_br:{optimizer.param_groups[0]['lr']}".upper())
            scheduler.step()
            print(f"lr_up:{optimizer.param_groups[0]['lr']}".upper())
            
    z_w_.append(err_real.item())
    if err_real.item()<high_rf:
            high_rf=err_real.item()
            torch.save(EFF_NET.state_dict(),f'/kaggle/working/{err_real.item()}.pth')
    if i==20:
        break
    i+=1

EPOCH : 0 LOSS_wl : 0.39007630944252014
EPOCH : 1 LOSS_wl : 0.3739895522594452
EPOCH : 2 LOSS_wl : 0.39566323161125183
EPOCH : 3 LOSS_wl : 0.33758997917175293
EPOCH : 4 LOSS_wl : 0.32837915420532227
EPOCH : 5 LOSS_wl : 0.4504753053188324
EPOCH : 6 LOSS_wl : 0.41383782029151917
EPOCH : 7 LOSS_wl : 0.3791745603084564
EPOCH : 8 LOSS_wl : 0.32800236344337463
LR_BR:0.0001
LR_UP:8.6E-05
EPOCH : 9 LOSS_wl : 0.409349650144577
EPOCH : 10 LOSS_wl : 0.39976009726524353
EPOCH : 11 LOSS_wl : 0.4801681637763977
EPOCH : 12 LOSS_wl : 0.44642773270606995
EPOCH : 13 LOSS_wl : 0.3950637876987457
EPOCH : 14 LOSS_wl : 0.4063793122768402
EPOCH : 15 LOSS_wl : 0.3240818381309509
EPOCH : 16 LOSS_wl : 0.34661516547203064
EPOCH : 17 LOSS_wl : 0.4195057451725006
EPOCH : 18 LOSS_wl : 0.3955972492694855
EPOCH : 19 LOSS_wl : 0.3462539613246918
EPOCH : 20 LOSS_wl : 0.4745922088623047
